# Lab 3: Compaction — SH Reduction, Pruning, and Size Analysis

**Objectives:**
- Implement SH degree reduction and measure its impact on color accuracy
- Sweep opacity pruning thresholds and plot survival curves
- Remove floater Gaussians via scale thresholding
- Build a combined pipeline and quantify cumulative compression
- Analyze attribute entropy to estimate quantization headroom


In [ ]:
!pip install -q numpy matplotlib scipy
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy
%matplotlib inline
print('Ready.')

## Setup: Synthetic 3DGS Data (N=50,000)

In [ ]:
np.random.seed(42)
N = 50_000
positions      = np.random.randn(N, 3) * 2.0
scales         = np.exp(np.random.randn(N, 3) * 0.5 - 2.0)
quats          = np.random.randn(N, 4); quats /= np.linalg.norm(quats, axis=1, keepdims=True)
opacity_logits = np.random.randn(N) * 2.0 + 1.0
opacities      = 1.0 / (1.0 + np.exp(-opacity_logits))
sh_dc          = np.random.rand(N, 3)
sh_ac          = np.random.randn(N, 45) * 0.1

def size_mb(n, n_params):
    return n * n_params * 4 / 1e6

baseline_mb = size_mb(N, 59)
print(f'Baseline: {N:,} Gaussians x 59 params x 4 bytes = {baseline_mb:.1f} MB')

## Section 1: SH Degree Reduction

Reducing SH degree is the highest-leverage compaction strategy (SH = 76% of params).

In [ ]:
def sh_params(degree):
    """Total SH params per channel for given max degree."""
    return (degree + 1) ** 2

def params_for_degree(degree):
    """Total params per Gaussian for given SH degree."""
    return 3 + 3 + 4 + 1 + sh_params(degree) * 3  # pos+scale+rot+opacity + SH*3ch

# Evaluate SH degree reduction
def color_rmse_for_degree(sh_dc, sh_ac, degree, n_views=100):
    """
    Approximate RMSE from dropping SH beyond `degree`.
    We compare color evaluated with full SH vs. truncated SH at random view directions.
    This is a simplified model: we measure magnitude of dropped coefficients.
    """
    if degree == 3:
        return 0.0
    # Coefficients per channel in full SH: DC(1) + AC(15) = 16
    # DC is the first 1 coeff, orders 1-3 are the AC coeffs
    # sh_ac has shape (N, 45) = 15 coeffs/channel * 3 channels
    # Coefficients to drop: everything after degree
    keep_ac = (sh_params(degree) - 1)  # AC coefficients to keep per channel
    dropped_ac = sh_ac[:, keep_ac*3:]   # dropped coefficients
    return float(np.sqrt(np.mean(dropped_ac**2)))

print(f'{"Degree":>7} {"SH coeff/ch":>12} {"Total params":>13} {"Size (MB)":>11} {"Ratio":>7} {"Color RMSE":>12}')
print('-' * 65)
for deg in [0, 1, 2, 3]:
    total_p = params_for_degree(deg)
    mb = size_mb(N, total_p)
    ratio = baseline_mb / mb
    rmse = color_rmse_for_degree(sh_dc, sh_ac, deg)
    print(f'{deg:>7} {sh_params(deg):>12} {total_p:>13} {mb:>10.1f}  {ratio:>6.2f}x {rmse:>12.6f}')

## Section 2: Opacity-Based Pruning

In [ ]:
thresholds = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
keep_fracs, keep_counts, sizes_mb = [], [], []

for t in thresholds:
    mask = opacities >= t
    n_kept = mask.sum()
    keep_fracs.append(n_kept / N)
    keep_counts.append(n_kept)
    sizes_mb.append(size_mb(n_kept, 59))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(thresholds, [f*100 for f in keep_fracs], 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Opacity threshold'); axes[0].set_ylabel('Gaussians kept (%)')
axes[0].set_title('Survival Rate vs. Threshold'); axes[0].grid(True, alpha=0.3)
axes[0].axvline(0.05, color='red', ls='--', label='typical=0.05'); axes[0].legend()

axes[1].semilogx(thresholds, sizes_mb, 's-', color='coral', lw=2)
axes[1].set_xlabel('Opacity threshold'); axes[1].set_ylabel('Size (MB)')
axes[1].set_title('File Size vs. Threshold'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'{"Threshold":>10} {"Kept":>10} {"Fraction":>10} {"Size (MB)":>11} {"Ratio":>8}')
print('-' * 52)
for t, n, mb in zip(thresholds, keep_counts, sizes_mb):
    print(f'{t:>10.3f} {n:>10,} {n/N:>10.3f} {mb:>10.1f}  {baseline_mb/mb:>6.2f}x')

## Section 3: Scale-Based Floater Removal

In [ ]:
max_scales = scales.max(axis=1)  # largest axis of each Gaussian
p99 = np.percentile(max_scales, 99)
p95 = np.percentile(max_scales, 95)

mask_scale = max_scales < p99
print(f'99th percentile scale: {p99:.4f}')
print(f'Gaussians removed (>p99): {(~mask_scale).sum():,} ({(~mask_scale).mean()*100:.1f}%)')
print(f'Remaining: {mask_scale.sum():,}')

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(np.log10(max_scales), bins=60, color='slateblue', alpha=0.8)
ax[0].axvline(np.log10(p99), color='red', ls='--', label=f'p99={p99:.3f}')
ax[0].axvline(np.log10(p95), color='orange', ls='--', label=f'p95={p95:.3f}')
ax[0].set_xlabel('log10(max scale)'); ax[0].set_title('Max Scale Distribution'); ax[0].legend()

ax[1].hist(np.log10(max_scales[mask_scale]), bins=60, color='green', alpha=0.8)
ax[1].set_xlabel('log10(max scale)'); ax[1].set_title('After Floater Removal')
plt.tight_layout(); plt.show()

## Section 4: Combined Compaction Pipeline

In [ ]:
# Stage 1: SH reduction to degree 1
sh_degree = 1
params_after_sh = params_for_degree(sh_degree)
size_after_sh = size_mb(N, params_after_sh)

# Stage 2: Opacity pruning at 0.05
threshold = 0.05
mask_opacity = opacities >= threshold
N_after_opacity = mask_opacity.sum()
size_after_opacity = size_mb(N_after_opacity, params_after_sh)

# Stage 3: Scale floater removal at p99
mask_scale_p99 = max_scales < np.percentile(max_scales, 99)
mask_combined = mask_opacity & mask_scale_p99
N_final = mask_combined.sum()
size_final = size_mb(N_final, params_after_sh)

print('Compaction Pipeline Results:')
print(f'{"Stage":<35} {"N Gaussians":>12} {"Params/G":>10} {"Size (MB)":>11} {"Ratio":>8}')
print('-' * 80)
print(f'{"Baseline (degree 3)":<35} {N:>12,} {59:>10} {baseline_mb:>10.1f}  {1.0:>6.2f}x')
print(f'{"After SH deg-1 reduction":<35} {N:>12,} {params_after_sh:>10} {size_after_sh:>10.1f}  {baseline_mb/size_after_sh:>6.2f}x')
print(f'{"After opacity pruning (>0.05)":<35} {N_after_opacity:>12,} {params_after_sh:>10} {size_after_opacity:>10.1f}  {baseline_mb/size_after_opacity:>6.2f}x')
print(f'{"After scale pruning (p99)":<35} {N_final:>12,} {params_after_sh:>10} {size_final:>10.1f}  {baseline_mb/size_final:>6.2f}x')

## Section 5: Attribute Entropy Analysis

Shannon entropy tells us the minimum bits needed per value. Attributes with low entropy are more compressible.

In [ ]:
def empirical_entropy_bits(values, n_bins=256):
    """Estimate Shannon entropy in bits via histogram."""
    hist, _ = np.histogram(values.flatten(), bins=n_bins)
    hist = hist[hist > 0]
    probs = hist / hist.sum()
    return float(scipy_entropy(probs, base=2))

attrs = {
    'Position X':      positions[:, 0],
    'Scale X (log)':   np.log(scales[:, 0]),
    'Opacity (logit)': opacity_logits,
    'DC Color R':      sh_dc[:, 0],
    'SH AC coeff 0':   sh_ac[:, 0],
    'SH AC coeff 10':  sh_ac[:, 10],
}
print(f'{"Attribute":<22} {"Entropy (bits)":>15} {"float32 uses":>14} {"Savings":>10}')
print('-' * 64)
for name, vals in attrs.items():
    h = empirical_entropy_bits(vals)
    savings = 32 / h
    print(f'{name:<22} {h:>15.2f} {32:>14} {savings:>9.1f}x')

print('\nNote: Lower entropy = more compressible. SH AC coeffs have lowest entropy.')
print('Entropy < 8 bits means 8-bit quantization + entropy coding can store without loss.')

## Key Takeaways

- **SH degree reduction** gives 2–3× parameter reduction with modest quality loss for diffuse scenes
- **Opacity pruning at 0.05** removes ~15–25% of Gaussians with negligible visual impact
- **Scale-based floater removal** at p99 removes <1% of Gaussians that would distort compression
- **Combined pipeline** achieves 3–5× from compaction alone, before any encoding
- **Entropy analysis** shows SH AC coefficients have <6 bits of entropy → 5-bit quantization loses almost nothing
- These compaction gains multiply with encoding gains (Module 5) — 3–5× × 6–7× ≈ 20–35× total
